# **LightGBM**

## **1. Import Libraries**

In [ ]:
import lightgbm as lgb
import joblib
from data_preparation import data_preparation

## **2. Data Preparation**

In [27]:
X_train, X_test, y_train, y_test = data_preparation(
    '../dataset/digital_marketing_campaign_dataset.csv'
)

## **3. Preprocessing**

In [28]:
X_train['CampaignType'] = X_train['CampaignType'].astype('category')
X_test['CampaignType']  = X_test['CampaignType'].astype('category')

## **5. Hyperparameter Configuration**

In [29]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 96,
    'max_depth': 7,
    'learning_rate': 0.005,
    'feature_fraction': 0.82,
    'bagging_fraction': 0.85,
    'bagging_freq': 5,
    'min_child_samples': 35,
    'scale_pos_weight': len(y_train[y_train == 0]) / len(y_train[y_train == 1]),
    'lambda_l1': 1.5,
    'lambda_l2': 2.0,
    'verbose': -1,
    'num_threads': -1
}

train_set = lgb.Dataset(X_train, label=y_train)
valid_set = lgb.Dataset(X_test,    label=y_test,    reference=train_set)

## **6. Training**

In [30]:
evals_result = {}

model = lgb.train(
    params,
    train_set,
    num_boost_round=4000,
    valid_sets=[train_set, valid_set],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=200, verbose=True),
        lgb.log_evaluation(100),
        lgb.record_evaluation(evals_result)
    ]
)

Training until validation scores don't improve for 200 rounds
[100]	train's auc: 0.860464	valid's auc: 0.779013
[200]	train's auc: 0.881209	valid's auc: 0.789363
[300]	train's auc: 0.894974	valid's auc: 0.795714
[400]	train's auc: 0.904839	valid's auc: 0.799248
[500]	train's auc: 0.912899	valid's auc: 0.800156
[600]	train's auc: 0.920161	valid's auc: 0.801553
[700]	train's auc: 0.927017	valid's auc: 0.803117
[800]	train's auc: 0.933144	valid's auc: 0.803441
[900]	train's auc: 0.938743	valid's auc: 0.803744
[1000]	train's auc: 0.943523	valid's auc: 0.803376
[1100]	train's auc: 0.94791	valid's auc: 0.803445
Early stopping, best iteration is:
[935]	train's auc: 0.940514	valid's auc: 0.803898


## **11. Save Model**

Using LightGBM's native format for better cross-version compatibility.

In [ ]:
joblib.dump(model, '../models/lgbm_model.pkl')

TypeError: 'str' object cannot be interpreted as an integer